# Guide-level analysis and density across the cell cycle, per gene target (OPS0077 KO cells)

The gene-level average of the CCNB1 / MCM3 / CENPC knockout cells shows no clear arrest against NTC. This notebook asks **whether individual guides differ** (some effective, some not) and shows the **cell density and marker profiles across the cell cycle for each gene**, in the style of `marker_profiles.png`.

It runs two scripts (`guide_level_analysis.py`, `plot_density_profiles_by_gene.py` in `scripts/ops0077_cc_transfer/`) and then explores their output. Prerequisites: the frozen NTC reference (`18_...`) and the scored knockout cells (`19_...`).

**A guide is the 20-nt protospacer (`obs['sgRNA']`).** The `barcode` column is truncated to 9 or 10 nucleotides, so using it would split one guide into two (4 guides per gene, not 8).

**Analyses**
1. per guide: change in the fraction of cells per sector against NTC, standardised to the guide's own well mix, with bootstrap intervals
2. do the guides of one gene differ more than chance? (permutation within wells)
3. is a guide effect real? The same guide is measured in two **disjoint sets of cells** (with a stain score vs without) by different instruments
4. do other phenotypes rank the guides the same way?
5. power, and density across the cycle per gene and per guide

In [ ]:
# %pip install anndata numpy pandas scipy scikit-learn matplotlib

## 1. Run the analyses

In [ ]:
import subprocess, sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.stats import spearmanr

ROOT = Path("..").resolve() if (Path("..") / "scripts").exists() else Path(".").resolve()
SCRIPTS = ROOT / "scripts/ops0077_cc_transfer"
SCORES = ROOT / "reports/ops0077_cc_transfer/ko3_scores/cc_phase_new_cells.csv"
GUIDE_DIR = ROOT / "reports/ops0077_cc_transfer/ko3_guide_analysis"
DENS_DIR = ROOT / "reports/ops0077_cc_transfer/ko3_density_profiles"
RUN = True            # False: only read existing results
assert SCORES.exists(), "run notebooks 18 and 19 first"
def run(cmd):
    p = subprocess.Popen([sys.executable, "-W", "ignore", *map(str, cmd)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    assert p.wait() == 0
if RUN:
    run([SCRIPTS / "guide_level_analysis.py", ROOT])
    run([SCRIPTS / "plot_density_profiles_by_gene.py", ROOT])

## 2. Density and marker profiles across the cycle, per gene
Top row: median and IQR of the phosphorylated Rb fraction, p21, pRb and DNA content along the phase (dashed: log nuclear area). Bottom row: density of cells along the phase; the grey step is the NTC density. Shading: NTC-defined sectors.

In [ ]:
display(Image(filename=str(DENS_DIR / "density_profiles_by_gene_stain_score.png")))
display(Image(filename=str(DENS_DIR / "density_profiles_by_gene_phase_contrast_score.png")))
pd.read_csv(DENS_DIR / "sector_fractions_by_gene.csv")

**Reading the plots.** The knockout cells follow the NTC profile (the DNA curve and the p21 / pRb / phosphorylated Rb pattern are the same), so the score is meaningful for them. The densities differ in *where* along the cycle cells accumulate: note the pile-up at the end of the loop (phase near 0.95, low DNA, high p21: just after division), strongest for CENPC. The phase-contrast version covers every cell but is less precise (it blurs the sharp peaks).

## 3. Guides: change in sector fractions against NTC (well-matched)

In [ ]:
gs = pd.read_csv(GUIDE_DIR / "guide_sector_shifts.csv")
first = gs.groupby("guide").first()
wide = gs.pivot(index="guide", columns="sector", values="delta")[["G1", "S", "G2/M", "M-to-G1 exit"]].round(3)
wide.insert(0, "n_stain_cells", first.loc[wide.index, "n_stain"].astype(int))
wide["perm_p"] = first.loc[wide.index, "perm_p"].round(4); wide["fdr_q"] = first.loc[wide.index, "fdr_q"].round(4)
wide  # delta = fraction of the guide's stain-scored cells minus the NTC fraction re-weighted to the guide's wells

In [ ]:
display(Image(filename=str(GUIDE_DIR / "g1_guide_shift_forest.png")))

Filled markers are the stain-scored cells (the precise measurement) with 95% bootstrap intervals; open squares use all cells, with the phase-contrast sector for cells without a stain score. `perm_p` is a permutation test of a shift of the mean phase against NTC, and `fdr_q` corrects for the 12 guides (Benjamini–Hochberg). Intervals are wide for guides with fewer than 60 scored cells.

In [ ]:
display(Image(filename=str(GUIDE_DIR / "g2_guide_density_along_cycle.png")))
display(Image(filename=str(GUIDE_DIR / "g6_guide_well_composition.png")))

## 4. Do guides of one gene differ more than chance?

In [ ]:
het = pd.read_csv(GUIDE_DIR / "heterogeneity_across_guides.csv"); display(het.round(4))
display(Image(filename=str(GUIDE_DIR / "g5_heterogeneity_tests.png")))

The chi-square compares the sector composition across the guides of a gene. The null distribution comes from shuffling guide labels **within wells**, so a different well mix between guides cannot explain a difference. `stain` uses the precise score on the scored cells; `all_cells_final` uses every cell (phase-contrast score where no stain score).

## 5. Is a guide effect real? Two disjoint sets of cells, two instruments

In [ ]:
conc = pd.read_csv(GUIDE_DIR / "guide_effect_concordance.csv"); display(conc.round(3))
display(Image(filename=str(GUIDE_DIR / "g3_guide_effect_concordance_disjoint_cells.png")))

If a guide's phase composition is a property of the guide, it should be reproduced in cells that were not used for the first estimate. Here the stain-scored cells (stain sectors) are compared with the **other** cells of the same guide (phase-contrast sectors). A high rank correlation across the 12 guides means the guide ranking is reproducible, not noise. The phase-contrast sectors have a different absolute calibration (they call fewer G2/M and more S cells), so only the *ranking* of the guides is compared, which is what Spearman measures. The narrow S sector is hard to measure and is expected to agree poorly.

## 6. Do other phenotypes rank the guides the same way?

In [ ]:
ph = pd.read_csv(GUIDE_DIR / "phenotypes_by_guide.csv", index_col=0); display(ph.round(2))
display(pd.read_csv(GUIDE_DIR / "phenotype_vs_g2m_shift_across_guides.csv", index_col=0).round(3))
display(Image(filename=str(GUIDE_DIR / "g4_guide_phenotypes_ranked_by_g2m_shift.png")))

**Caution when reading the correlations.** DNA content and the phosphorylated Rb fraction are *inputs* of the score, so their agreement with the G2/M shift is expected by construction and is not independent confirmation. Nuclear area and p21 are less tied to the G2/M call.

## 7. A shared shift: the end of the loop (phase ≥ 0.92)
Cells just after division have low DNA and high p21. Is this fraction higher in every knockout than in NTC? Is it a guide effect or a gene effect?

In [ ]:
ko = pd.read_csv(SCORES, index_col=0); ko["guide"] = ko["gene"] + "|" + ko["sgRNA"].astype(str).str[:8]
ntc = pd.read_csv(ROOT / "reports/ops0077_cc_transfer/reference/ntc_reference_scores.csv", index_col=0).dropna(subset=["cc_phase_stain"])
CUT = 0.92; rng = np.random.default_rng(0)
nw = {w: (d.cc_phase_stain >= CUT).mean() for w, d in ntc.groupby("well")}
def stat(d):
    f = (d.cc_phase_stain >= CUT).values; comp = d.well.value_counts(normalize=True)
    return f.mean(), f.mean() - sum(comp[w] * nw[w] for w in comp.index)
ks = ko.dropna(subset=["cc_phase_stain"]); rows = []
for key, d in list(ks.groupby("gene")) + list(ks.groupby("guide")):
    f, dl = stat(d); b = [stat(d.iloc[rng.integers(0, len(d), len(d))])[1] for _ in range(1000)]
    rows.append(dict(group=key, n=len(d), fraction=round(f, 3), delta_vs_ntc_well_matched=round(dl, 3), ci_low=round(np.percentile(b, 2.5), 3), ci_high=round(np.percentile(b, 97.5), 3)))
print("NTC fraction:", round((ntc.cc_phase_stain >= CUT).mean(), 3)); pd.DataFrame(rows)

All three genes have more cells at the end of the loop than NTC, but at guide level it is driven mostly by a few guides. A shift shared by different targets is not specific to the targeted gene. One hypothesis, **not tested here**, is a general response of cells in which Cas9 cut the DNA (a p53 / p21 DNA-damage response), which non-targeting guides do not trigger; a cutting control (for example intergenic or safe-targeting guides) would test it. Guide efficiency, the time between transduction and fixation, and partial or mixed knockouts are other explanations.

## 8. Statistical power
How big a change can a guide of n cells reveal? (half-width of a 95% interval on a fraction)

In [ ]:
pd.read_csv(GUIDE_DIR / "power_by_guide.csv", index_col=0)

## 9. Output files

| folder | content |
|---|---|
| `reports/ops0077_cc_transfer/ko3_guide_analysis/` | `guide_sector_shifts.csv`, `heterogeneity_across_guides.csv`, `guide_effect_concordance.csv`, `phenotypes_by_guide.csv`, `power_by_guide.csv`, figures `g1` to `g6` |
| `reports/ops0077_cc_transfer/ko3_density_profiles/` | density and marker-profile figures per gene (stain and phase-contrast score) and `sector_fractions_by_gene.csv` |